# RQ3: LLM-as-a-Judge Report Evaluation (v4)

This notebook analyses the incremental outputs of `judge_reference_alignment_v4.py`
and makes no API calls. The five primary outcomes are question relevance,
key-point alignment (KPA), key-point contradiction rate (KPC), conclusion
alignment, and uncertainty alignment. Model comparisons are paired by PMID.



In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "gpt_researcher").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")

EVAL_DIR = REPOSITORY_ROOT / "evaluation" / "reports"
OUT = REPOSITORY_ROOT / "outputs" / "evaluation" / "report_alignment"
RESULTS = OUT / "judge_alignment.jsonl"
ERRORS = OUT / "judge_alignment_errors.jsonl"
REFERENCES = REPOSITORY_ROOT / "benchmark" / "reference_keypoints.jsonl"
DEV_PMIDS = REPOSITORY_ROOT / "benchmark" / "benchmark.json"
ANALYSIS = OUT / "analysis"
FIGURES, TABLES = ANALYSIS / "figures", ANALYSIS / "tables"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

SPLIT = "development"  # Change to "test" or "all" later.
PAIRED_ONLY = True
PROMPT_VERSION = "judge-alignment-v4"
N_BOOT, SEED = 10_000, 2026

MODELS = ["Qwen3-8B", "Fleming-R1-7B"]
MODEL_COLORS = {"Qwen3-8B": "#176B87", "Fleming-R1-7B": "#D97706"}
QUESTION_ORDER = ["direct", "partial", "off_topic"]
STATUS_ORDER = ["covered", "partially_covered", "missing", "contradicted"]
CONCLUSION_ORDER = ["aligned", "partially_aligned", "not_addressed", "contradicted"]
UNCERTAINTY_ORDER = [
    "matched",
    "stronger_than_reference",
    "weaker_than_reference",
    "not_assessable",
]
QUESTION_SCORE = {"direct": 1.0, "partial": 0.5, "off_topic": 0.0}
STATUS_SCORE = {
    "covered": 1.0,
    "partially_covered": 0.5,
    "missing": 0.0,
    "contradicted": 0.0,
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "semibold",
    }
)
print("Evaluation directory:", EVAL_DIR)



## 1. Load and flatten the incremental results

The newest successful v4 record is retained for each model and PMID. Repeated
error entries are collapsed, and errors followed by a valid result are resolved.



In [ ]:
def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding="utf-8") as handle:
        for number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"Skipping malformed JSON at {path.name}:{number}: {exc}")
    return rows


def load_pmid_set(path):
    if not path.exists():
        return set()
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, dict):
        data = data.get("reviews", data.get("items", data.get("pmids", [])))
    return {
        str(item.get("pmid") if isinstance(item, dict) else item)
        for item in data
    }


def exact_key(row):
    return (
        str(row.get("subject_model", "")),
        str(row.get("pmid", "")),
        str(row.get("report_sha256", "")),
        str(row.get("reference_sha256", "")),
        str(row.get("prompt_version", "")),
    )


success_raw = [
    row
    for row in read_jsonl(RESULTS)
    if row.get("status") == "ok" and row.get("prompt_version") == PROMPT_VERSION
]
error_raw = [
    row
    for row in read_jsonl(ERRORS)
    if row.get("status") == "error" and row.get("prompt_version") == PROMPT_VERSION
]
reference_raw = [row for row in read_jsonl(REFERENCES) if row.get("status") == "ok"]
development_pmids = load_pmid_set(DEV_PMIDS)
reference_pmids = {str(row["pmid"]) for row in reference_raw}

latest = {}
for row in sorted(success_raw, key=lambda item: item.get("created_at", "")):
    latest[(str(row["subject_model"]), str(row["pmid"]))] = row
records = list(latest.values())

successful_keys = {exact_key(row) for row in success_raw}
unresolved = {}
for row in sorted(error_raw, key=lambda item: item.get("created_at", "")):
    if exact_key(row) not in successful_keys:
        unresolved[exact_key(row)] = row
unresolved_errors = list(unresolved.values())

print("Current successful model-PMID evaluations:", len(records))
print("Unresolved evaluation errors:", len(unresolved_errors))
print("Reference PMIDs:", len(reference_pmids))



In [ ]:
def safe_mean(values):
    values = [float(value) for value in values if pd.notna(value)]
    return float(np.mean(values)) if values else np.nan


review_rows, keypoint_rows = [], []
for record in records:
    pmid, model = str(record["pmid"]), str(record["subject_model"])
    references_by_id = {
        str(point["id"]): point for point in record.get("reference_key_points", [])
    }
    assessments = record.get("key_point_assessments", [])
    scores, core_scores, supporting_scores, conflicts, statuses = [], [], [], [], []

    for item in assessments:
        keypoint_id = str(item["key_point_id"])
        reference = references_by_id.get(keypoint_id, {})
        status = str(item.get("status", "")).lower()
        score = STATUS_SCORE.get(status, np.nan)
        importance = str(reference.get("importance", "unknown")).lower()
        conflict = bool(item.get("conflicting_components")) or status == "contradicted"
        scores.append(score)
        conflicts.append(float(conflict))
        statuses.append(status)
        if importance == "core":
            core_scores.append(score)
        elif importance == "supporting":
            supporting_scores.append(score)
        keypoint_rows.append(
            {
                "pmid": pmid,
                "split": record.get("split"),
                "subject_model": model,
                "key_point_id": keypoint_id,
                "importance": importance,
                "category": str(reference.get("category", "unknown")).lower(),
                "reference_statement": reference.get("statement", ""),
                "status": status,
                "score": score,
                "difference_type": item.get("difference_type"),
                "has_conflict": conflict,
                "aligned_components": item.get("aligned_components") or [],
                "missing_components": item.get("missing_components") or [],
                "conflicting_components": item.get("conflicting_components") or [],
                "report_evidence": item.get("report_evidence") or [],
                "justification": item.get("justification", ""),
                "report_path": record.get("report_path", ""),
            }
        )

    question = record.get("question_relevance") or {}
    conclusion = record.get("conclusion_alignment") or {}
    uncertainty = record.get("uncertainty_alignment") or {}
    review_rows.append(
        {
            "pmid": pmid,
            "split": record.get("split"),
            "subject_model": model,
            "question_relevance": question.get("label"),
            "question_relevance_score": QUESTION_SCORE.get(question.get("label"), np.nan),
            "key_point_alignment": safe_mean(scores),
            "core_alignment": safe_mean(core_scores),
            "supporting_alignment": safe_mean(supporting_scores),
            "key_point_contradiction_rate": safe_mean(conflicts),
            "n_key_points": len(scores),
            "covered": statuses.count("covered"),
            "partially_covered": statuses.count("partially_covered"),
            "missing": statuses.count("missing"),
            "contradicted": statuses.count("contradicted"),
            "conclusion_alignment": conclusion.get("label"),
            "conclusion_difference_type": conclusion.get("difference_type"),
            "uncertainty_alignment": uncertainty.get("label"),
            "attempts": record.get("attempts"),
            "report_path": record.get("report_path", ""),
        }
    )

reviews_all = pd.DataFrame(review_rows)
keypoints_all = pd.DataFrame(keypoint_rows)
if reviews_all.empty:
    raise RuntimeError("No successful v4 evaluations were found.")
print("Flattened reviews:", len(reviews_all))
print("Flattened atomic key points:", len(keypoints_all))



## 2. Evaluation coverage and paired cohort

Progress and unresolved validation errors are reported separately. The main
comparative cohort retains only PMIDs successfully judged for both models.



In [ ]:
def expected_for_split(split):
    if split == "development":
        return reference_pmids & development_pmids
    if split == "test":
        return reference_pmids - development_pmids
    return reference_pmids


expected_pmids = expected_for_split(SPLIT)
split_reviews = reviews_all if SPLIT == "all" else reviews_all[reviews_all["split"] == SPLIT]
split_keypoints = keypoints_all if SPLIT == "all" else keypoints_all[keypoints_all["split"] == SPLIT]
model_pmids = {
    model: set(split_reviews.loc[split_reviews["subject_model"] == model, "pmid"])
    for model in MODELS
}
paired_pmids = (
    set.intersection(*(model_pmids[model] for model in MODELS))
    if all(model_pmids[model] for model in MODELS)
    else set()
)
analysis_reviews = (
    split_reviews[split_reviews["pmid"].isin(paired_pmids)].copy()
    if PAIRED_ONLY
    else split_reviews.copy()
)
analysis_keypoints = (
    split_keypoints[split_keypoints["pmid"].isin(paired_pmids)].copy()
    if PAIRED_ONLY
    else split_keypoints.copy()
)

progress = pd.DataFrame(
    [
        {
            "Model": model,
            "Expected PMIDs": len(expected_pmids),
            "Successful evaluations": len(model_pmids[model]),
            "Completion (%)": 100 * len(model_pmids[model]) / len(expected_pmids)
            if expected_pmids
            else np.nan,
            "Included paired PMIDs": len(model_pmids[model] & paired_pmids),
        }
        for model in MODELS
    ]
).set_index("Model")
display(progress.style.format({"Completion (%)": "{:.1f}"}))

error_frame = pd.DataFrame(unresolved_errors)
if not error_frame.empty and SPLIT != "all":
    error_frame = error_frame[error_frame["split"] == SPLIT]
if not error_frame.empty:
    error_summary = (
        error_frame.groupby(["subject_model", "error"], dropna=False)
        .size()
        .rename("Unresolved cases")
        .reset_index()
    )
    display(Markdown("**Unresolved judge errors**"))
    display(error_summary)
    error_summary.to_csv(TABLES / f"unresolved_errors_{SPLIT}.csv", index=False)
else:
    print("No unresolved judge errors in this split.")
print("Paired PMIDs:", len(paired_pmids))
print("Reports in analytical cohort:", len(analysis_reviews))
progress.to_csv(TABLES / f"evaluation_progress_{SPLIT}.csv")



## 3. Primary review-level metrics

KPA assigns 1 to covered, 0.5 to partially covered, and 0 to missing or
contradicted points. KPC counts any point with conflicting components. Higher
values are better except for KPC. Bootstrap intervals reflect variation across
reviews, not repeated model executions.



In [ ]:
def bootstrap_mean(values):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if not len(values):
        return np.nan, np.nan, np.nan
    estimate = float(values.mean())
    if len(values) == 1:
        return estimate, np.nan, np.nan
    rng = np.random.default_rng(SEED)
    samples = values[rng.integers(0, len(values), size=(N_BOOT, len(values)))].mean(axis=1)
    low, high = np.percentile(samples, [2.5, 97.5])
    return estimate, float(low), float(high)


METRICS = {
    "question_relevance_score": "Question relevance",
    "key_point_alignment": "Key-point alignment",
    "core_alignment": "Core key-point alignment",
    "key_point_contradiction_rate": "Key-point contradiction rate",
}
summary_rows = []
for model in MODELS:
    model_data = analysis_reviews[analysis_reviews["subject_model"] == model]
    for column, label in METRICS.items():
        mean, low, high = bootstrap_mean(model_data[column])
        summary_rows.append(
            {
                "Model": model,
                "Metric": label,
                "N reviews": int(model_data[column].notna().sum()),
                "Mean": mean,
                "CI 95% low": low,
                "CI 95% high": high,
            }
        )
primary = pd.DataFrame(summary_rows)
formatted = primary.copy()
formatted["Mean [95% CI]"] = formatted.apply(
    lambda row: f"{row['Mean']:.3f} [{row['CI 95% low']:.3f}, {row['CI 95% high']:.3f}]"
    if pd.notna(row["CI 95% low"])
    else f"{row['Mean']:.3f}",
    axis=1,
)
primary_table = formatted.pivot(
    index="Model", columns="Metric", values="Mean [95% CI]"
).reindex(MODELS)
display(primary_table)
primary.to_csv(TABLES / f"primary_metrics_{SPLIT}.csv", index=False)
(TABLES / f"primary_metrics_{SPLIT}.tex").write_text(
    primary_table.to_latex(escape=False), encoding="utf-8"
)



In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.7), sharey=True)
for axis, label in zip(axes, METRICS.values()):
    data = primary[primary["Metric"] == label].set_index("Model")
    means = np.array([data.loc[model, "Mean"] for model in MODELS])
    lows = np.array([data.loc[model, "CI 95% low"] for model in MODELS])
    highs = np.array([data.loc[model, "CI 95% high"] for model in MODELS])
    positions = np.arange(len(MODELS))
    axis.bar(positions, means, width=0.65, color=[MODEL_COLORS[m] for m in MODELS])
    if np.isfinite(lows).all() and np.isfinite(highs).all():
        axis.errorbar(
            positions,
            means,
            yerr=np.vstack([means - lows, highs - means]),
            fmt="none",
            ecolor="#333333",
            capsize=4,
        )
    axis.set_xticks(positions)
    axis.set_xticklabels(MODELS, rotation=20, ha="right")
    axis.set_title(label)
    axis.set_ylim(0, 1)
    axis.grid(axis="y", alpha=0.22)
axes[0].set_ylabel("Macro mean")
fig.suptitle(f"Primary LLM-judge metrics on the {SPLIT} cohort", y=1.03)
fig.tight_layout()
fig.savefig(FIGURES / f"primary_metrics_{SPLIT}.pdf", bbox_inches="tight")
fig.savefig(FIGURES / f"primary_metrics_{SPLIT}.png", bbox_inches="tight", dpi=220)
plt.show()



## 4. Paired Qwen-Fleming differences

Each difference is Qwen3-8B minus Fleming-R1-7B for the same PMID. Positive
differences favour Qwen for relevance and alignment; negative differences favour
Qwen for contradiction rate.



In [ ]:
def paired_difference(metric):
    pivot = analysis_reviews.pivot(index="pmid", columns="subject_model", values=metric)
    if not all(model in pivot.columns for model in MODELS):
        return None
    paired = pivot[MODELS].dropna()
    if paired.empty:
        return None
    delta = (paired["Qwen3-8B"] - paired["Fleming-R1-7B"]).to_numpy()
    rng = np.random.default_rng(SEED)
    samples = delta[rng.integers(0, len(delta), size=(N_BOOT, len(delta)))].mean(axis=1)
    low, high = np.percentile(samples, [2.5, 97.5])
    return {
        "Metric": METRICS[metric],
        "N paired": len(delta),
        "Qwen mean": paired["Qwen3-8B"].mean(),
        "Fleming mean": paired["Fleming-R1-7B"].mean(),
        "Mean difference": delta.mean(),
        "CI 95% low": low,
        "CI 95% high": high,
        "Qwen higher": int((delta > 1e-12).sum()),
        "Equal": int((abs(delta) <= 1e-12).sum()),
        "Qwen lower": int((delta < -1e-12).sum()),
    }


paired_summary = pd.DataFrame(
    [row for row in (paired_difference(metric) for metric in METRICS) if row is not None]
)
if paired_summary.empty:
    print("Paired comparison not yet available.")
else:
    display(
        paired_summary.style.format(
            {
                "Qwen mean": "{:.3f}",
                "Fleming mean": "{:.3f}",
                "Mean difference": "{:+.3f}",
                "CI 95% low": "{:+.3f}",
                "CI 95% high": "{:+.3f}",
            }
        )
    )
    paired_summary.to_csv(TABLES / f"paired_differences_{SPLIT}.csv", index=False)



## 5. Atomic key-point outcomes

This micro-averaged distribution separates omission from contradiction. KPC can
exceed the contradicted percentage because partially covered points may contain
conflicting components.



In [ ]:
status_counts = (
    analysis_keypoints.groupby(["subject_model", "status"])
    .size()
    .reindex(
        pd.MultiIndex.from_product(
            [MODELS, STATUS_ORDER], names=["subject_model", "status"]
        ),
        fill_value=0,
    )
    .rename("Count")
    .reset_index()
)
status_counts["Percentage"] = status_counts.groupby("subject_model")["Count"].transform(
    lambda values: 100 * values / values.sum() if values.sum() else 0
)
status_table = status_counts.pivot(
    index="subject_model", columns="status", values="Percentage"
).reindex(index=MODELS, columns=STATUS_ORDER).fillna(0)
display(status_table.style.format("{:.1f}%"))
status_counts.to_csv(TABLES / f"keypoint_status_{SPLIT}.csv", index=False)

status_colors = {
    "covered": "#2F855A",
    "partially_covered": "#E0A126",
    "missing": "#9CA3AF",
    "contradicted": "#C2413B",
}
fig, axis = plt.subplots(figsize=(8.5, 3.4))
left = np.zeros(len(MODELS))
for status in STATUS_ORDER:
    values = status_table[status].to_numpy()
    axis.barh(
        MODELS,
        values,
        left=left,
        color=status_colors[status],
        label=status.replace("_", " ").title(),
    )
    for index, (start, value) in enumerate(zip(left, values)):
        if value >= 6:
            axis.text(
                start + value / 2,
                index,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color="white" if status in {"covered", "contradicted"} else "#222222",
            )
    left += values
axis.set_xlim(0, 100)
axis.set_xlabel("Reference key points (%)")
axis.set_title(f"Reference key-point outcomes on the {SPLIT} cohort")
axis.grid(axis="x", alpha=0.2)
axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22), frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / f"keypoint_status_{SPLIT}.pdf", bbox_inches="tight")
fig.savefig(FIGURES / f"keypoint_status_{SPLIT}.png", bbox_inches="tight", dpi=220)
plt.show()



### Key-point type diagnostics

Scores are first averaged within each review and category, avoiding excessive
weight for reviews with several points of the same type.



In [ ]:
category_review = analysis_keypoints.groupby(
    ["subject_model", "pmid", "category"], as_index=False
).agg(alignment=("score", "mean"), conflict_rate=("has_conflict", "mean"))
category_summary = category_review.groupby(
    ["subject_model", "category"], as_index=False
).agg(
    n_reviews=("pmid", "nunique"),
    mean_alignment=("alignment", "mean"),
    mean_conflict_rate=("conflict_rate", "mean"),
)
importance_review = analysis_keypoints.groupby(
    ["subject_model", "pmid", "importance"], as_index=False
).agg(alignment=("score", "mean"), conflict_rate=("has_conflict", "mean"))
importance_summary = importance_review.groupby(
    ["subject_model", "importance"], as_index=False
).agg(
    n_reviews=("pmid", "nunique"),
    mean_alignment=("alignment", "mean"),
    mean_conflict_rate=("conflict_rate", "mean"),
)
display(Markdown("**By category**"))
display(
    category_summary.style.format(
        {"mean_alignment": "{:.3f}", "mean_conflict_rate": "{:.3f}"}
    )
)
display(Markdown("**By importance**"))
display(
    importance_summary.style.format(
        {"mean_alignment": "{:.3f}", "mean_conflict_rate": "{:.3f}"}
    )
)
category_summary.to_csv(TABLES / f"alignment_by_category_{SPLIT}.csv", index=False)
importance_summary.to_csv(TABLES / f"alignment_by_importance_{SPLIT}.csv", index=False)

categories = sorted(category_summary["category"].dropna().unique())
if categories:
    fig, axis = plt.subplots(figsize=(max(7.5, 1.25 * len(categories)), 4.1))
    positions, width = np.arange(len(categories)), 0.36
    for offset, model in zip([-width / 2, width / 2], MODELS):
        data = category_summary[category_summary["subject_model"] == model].set_index(
            "category"
        )
        values = [
            data.loc[category, "mean_alignment"] if category in data.index else np.nan
            for category in categories
        ]
        axis.bar(
            positions + offset,
            values,
            width,
            color=MODEL_COLORS[model],
            label=model,
        )
    axis.set_xticks(positions)
    axis.set_xticklabels(
        [category.replace("_", " ").title() for category in categories],
        rotation=25,
        ha="right",
    )
    axis.set_ylim(0, 1)
    axis.set_ylabel("Mean alignment")
    axis.set_title(f"Alignment by key-point category ({SPLIT})")
    axis.grid(axis="y", alpha=0.22)
    axis.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURES / f"alignment_by_category_{SPLIT}.pdf", bbox_inches="tight")
    fig.savefig(
        FIGURES / f"alignment_by_category_{SPLIT}.png", bbox_inches="tight", dpi=220
    )
    plt.show()



## 6. Relevance, conclusion, and uncertainty distributions

Conclusion and uncertainty remain categorical. Overconfidence is calculated only
among assessable reports: stronger / (matched + stronger + weaker).



In [ ]:
def categorical_distribution(column, order):
    data = (
        analysis_reviews.groupby(["subject_model", column])
        .size()
        .reindex(
            pd.MultiIndex.from_product(
                [MODELS, order], names=["subject_model", column]
            ),
            fill_value=0,
        )
        .rename("Count")
        .reset_index()
    )
    data["Percentage"] = data.groupby("subject_model")["Count"].transform(
        lambda values: 100 * values / values.sum() if values.sum() else 0
    )
    return data


question_distribution = categorical_distribution("question_relevance", QUESTION_ORDER)
conclusion_distribution = categorical_distribution(
    "conclusion_alignment", CONCLUSION_ORDER
)
uncertainty_distribution = categorical_distribution(
    "uncertainty_alignment", UNCERTAINTY_ORDER
)
for title, data, column, order in [
    ("Question relevance", question_distribution, "question_relevance", QUESTION_ORDER),
    (
        "Conclusion alignment",
        conclusion_distribution,
        "conclusion_alignment",
        CONCLUSION_ORDER,
    ),
    (
        "Uncertainty alignment",
        uncertainty_distribution,
        "uncertainty_alignment",
        UNCERTAINTY_ORDER,
    ),
]:
    display(Markdown(f"**{title} (%)**"))
    display(
        data.pivot(index="subject_model", columns=column, values="Percentage")
        .reindex(index=MODELS, columns=order)
        .style.format("{:.1f}%")
    )

overconfidence_rows = []
for model in MODELS:
    data = analysis_reviews[analysis_reviews["subject_model"] == model]
    assessable = data[
        data["uncertainty_alignment"].isin(
            ["matched", "stronger_than_reference", "weaker_than_reference"]
        )
    ]
    stronger = int(
        (assessable["uncertainty_alignment"] == "stronger_than_reference").sum()
    )
    overconfidence_rows.append(
        {
            "Model": model,
            "Assessable reports": len(assessable),
            "Stronger than reference": stronger,
            "Overconfidence rate": stronger / len(assessable)
            if len(assessable)
            else np.nan,
        }
    )
overconfidence = pd.DataFrame(overconfidence_rows).set_index("Model")
display(Markdown("**Overconfidence among assessable reports**"))
display(overconfidence.style.format({"Overconfidence rate": "{:.3f}"}))
question_distribution.to_csv(TABLES / f"question_relevance_{SPLIT}.csv", index=False)
conclusion_distribution.to_csv(
    TABLES / f"conclusion_alignment_{SPLIT}.csv", index=False
)
uncertainty_distribution.to_csv(
    TABLES / f"uncertainty_alignment_{SPLIT}.csv", index=False
)
overconfidence.to_csv(TABLES / f"overconfidence_{SPLIT}.csv")



In [ ]:
distribution_specs = [
    (
        question_distribution,
        "question_relevance",
        QUESTION_ORDER,
        {"direct": "#2F855A", "partial": "#E0A126", "off_topic": "#C2413B"},
        "Question relevance",
    ),
    (
        conclusion_distribution,
        "conclusion_alignment",
        CONCLUSION_ORDER,
        {
            "aligned": "#2F855A",
            "partially_aligned": "#E0A126",
            "not_addressed": "#9CA3AF",
            "contradicted": "#C2413B",
        },
        "Conclusion alignment",
    ),
    (
        uncertainty_distribution,
        "uncertainty_alignment",
        UNCERTAINTY_ORDER,
        {
            "matched": "#2F855A",
            "stronger_than_reference": "#C2413B",
            "weaker_than_reference": "#4C78A8",
            "not_assessable": "#9CA3AF",
        },
        "Uncertainty alignment",
    ),
]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for axis, (data, column, order, colors, title) in zip(axes, distribution_specs):
    table = (
        data.pivot(index="subject_model", columns=column, values="Percentage")
        .reindex(index=MODELS, columns=order)
        .fillna(0)
    )
    left = np.zeros(len(MODELS))
    for category in order:
        values = table[category].to_numpy()
        axis.barh(
            MODELS,
            values,
            left=left,
            color=colors[category],
            label=category.replace("_", " ").title(),
        )
        left += values
    axis.set_xlim(0, 100)
    axis.set_xlabel("Reports (%)")
    axis.set_title(title)
    axis.grid(axis="x", alpha=0.2)
    axis.legend(
        fontsize=8,
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),
    )
fig.suptitle(f"Categorical LLM-judge outcomes on the {SPLIT} cohort", y=1.03)
fig.tight_layout()
fig.savefig(FIGURES / f"categorical_outcomes_{SPLIT}.pdf", bbox_inches="tight")
fig.savefig(
    FIGURES / f"categorical_outcomes_{SPLIT}.png", bbox_inches="tight", dpi=220
)
plt.show()



## 7. Diagnostics and manual-review queue

Difference types and paired cross-tabulations explain disagreement. The manual
queue prioritises low-relevance, low-core-alignment, contradictory, and
overconfident reports. These are diagnostics rather than headline metrics.



In [ ]:
keypoint_differences = (
    analysis_keypoints[analysis_keypoints["difference_type"] != "none"]
    .groupby(["subject_model", "difference_type"])
    .size()
    .rename("Key points")
    .reset_index()
)
conclusion_differences = (
    analysis_reviews[analysis_reviews["conclusion_difference_type"] != "none"]
    .groupby(["subject_model", "conclusion_difference_type"])
    .size()
    .rename("Reports")
    .reset_index()
)
display(Markdown("**Key-point difference types**"))
display(keypoint_differences)
display(Markdown("**Conclusion difference types**"))
display(conclusion_differences)
keypoint_differences.to_csv(
    TABLES / f"keypoint_difference_types_{SPLIT}.csv", index=False
)
conclusion_differences.to_csv(
    TABLES / f"conclusion_difference_types_{SPLIT}.csv", index=False
)


def paired_crosstab(column):
    pivot = analysis_reviews.pivot(
        index="pmid", columns="subject_model", values=column
    )
    if not all(model in pivot.columns for model in MODELS):
        return pd.DataFrame()
    pivot = pivot[MODELS].dropna()
    return pd.crosstab(
        pivot["Qwen3-8B"],
        pivot["Fleming-R1-7B"],
        rownames=["Qwen3-8B"],
        colnames=["Fleming-R1-7B"],
    )


conclusion_cross = paired_crosstab("conclusion_alignment")
uncertainty_cross = paired_crosstab("uncertainty_alignment")
display(Markdown("**Paired conclusion labels**"))
display(conclusion_cross)
display(Markdown("**Paired uncertainty labels**"))
display(uncertainty_cross)
conclusion_cross.to_csv(TABLES / f"paired_conclusion_crosstab_{SPLIT}.csv")
uncertainty_cross.to_csv(TABLES / f"paired_uncertainty_crosstab_{SPLIT}.csv")

manual = analysis_reviews.copy()
manual["priority"] = (
    3 * manual["question_relevance"].isin(["partial", "off_topic"]).astype(int)
    + 3
    * manual["conclusion_alignment"]
    .isin(["contradicted", "not_addressed"])
    .astype(int)
    + 2 * (manual["key_point_contradiction_rate"] > 0).astype(int)
    + 2 * (manual["core_alignment"] < 0.5).astype(int)
    + (manual["uncertainty_alignment"] == "stronger_than_reference").astype(int)
)
manual = manual.sort_values(
    ["priority", "core_alignment", "key_point_contradiction_rate"],
    ascending=[False, True, False],
)
manual_columns = [
    "pmid",
    "subject_model",
    "priority",
    "question_relevance",
    "key_point_alignment",
    "core_alignment",
    "key_point_contradiction_rate",
    "conclusion_alignment",
    "uncertainty_alignment",
    "report_path",
]
display(Markdown("**Highest-priority manual checks**"))
display(manual[manual_columns].head(20))
manual[manual_columns].to_csv(
    TABLES / f"manual_review_queue_{SPLIT}.csv", index=False
)
analysis_keypoints[analysis_keypoints["has_conflict"]].to_csv(
    TABLES / f"conflicting_keypoints_{SPLIT}.csv", index=False
)
print("Tables:", TABLES)
print("Figures:", FIGURES)



## 8. Human validation sample for the judge

A fixed random sample of 10 paired PMIDs corresponds to 20 reports. The two CSV
templates are created only once, so rerunning the notebook will not overwrite
manual annotations. Fill the human columns after independently checking the
candidate report against the reference key points. This validates the judge; it
is not an additional measure of report quality.



In [ ]:
validation_review_path = TABLES / f"human_judge_validation_reviews_{SPLIT}.csv"
validation_keypoint_path = TABLES / f"human_judge_validation_keypoints_{SPLIT}.csv"

if not validation_review_path.exists() or not validation_keypoint_path.exists():
    rng = np.random.default_rng(SEED)
    available = np.array(sorted(paired_pmids))
    sample_size = min(10, len(available))
    sampled_pmids = set(rng.choice(available, size=sample_size, replace=False))

    review_validation = analysis_reviews[
        analysis_reviews["pmid"].isin(sampled_pmids)
    ][
        [
            "pmid",
            "subject_model",
            "question_relevance",
            "conclusion_alignment",
            "uncertainty_alignment",
            "report_path",
        ]
    ].rename(
        columns={
            "question_relevance": "judge_question_relevance",
            "conclusion_alignment": "judge_conclusion_alignment",
            "uncertainty_alignment": "judge_uncertainty_alignment",
        }
    )
    review_validation["human_question_relevance"] = ""
    review_validation["human_conclusion_alignment"] = ""
    review_validation["human_uncertainty_alignment"] = ""
    review_validation["human_notes"] = ""
    review_validation.to_csv(validation_review_path, index=False)

    keypoint_validation = analysis_keypoints[
        analysis_keypoints["pmid"].isin(sampled_pmids)
    ][
        [
            "pmid",
            "subject_model",
            "key_point_id",
            "importance",
            "category",
            "reference_statement",
            "status",
            "has_conflict",
            "report_evidence",
            "justification",
            "report_path",
        ]
    ].rename(columns={"status": "judge_status", "has_conflict": "judge_has_conflict"})
    keypoint_validation["human_status"] = ""
    keypoint_validation["human_has_conflict"] = ""
    keypoint_validation["human_notes"] = ""
    keypoint_validation.to_csv(validation_keypoint_path, index=False)
    print(f"Created validation templates for {sample_size} paired PMIDs.")
else:
    print("Existing human-validation templates were preserved.")

display(Markdown(f"Review template: `{validation_review_path}`"))
display(Markdown(f"Key-point template: `{validation_keypoint_path}`"))


def agreement_if_complete(path, pairs):
    frame = pd.read_csv(path, dtype=str).fillna("")
    rows = []
    for judge_column, human_column, label in pairs:
        valid = frame[human_column].str.strip().ne("")
        if valid.any():
            exact = (
                frame.loc[valid, judge_column].str.strip().str.lower()
                == frame.loc[valid, human_column].str.strip().str.lower()
            )
            rows.append(
                {
                    "Decision": label,
                    "Manually assessed items": int(valid.sum()),
                    "Exact agreement": float(exact.mean()),
                }
            )
    return pd.DataFrame(rows)


review_agreement = agreement_if_complete(
    validation_review_path,
    [
        (
            "judge_question_relevance",
            "human_question_relevance",
            "Question relevance",
        ),
        (
            "judge_conclusion_alignment",
            "human_conclusion_alignment",
            "Conclusion alignment",
        ),
        (
            "judge_uncertainty_alignment",
            "human_uncertainty_alignment",
            "Uncertainty alignment",
        ),
    ],
)
keypoint_agreement = agreement_if_complete(
    validation_keypoint_path,
    [
        ("judge_status", "human_status", "Key-point status"),
        ("judge_has_conflict", "human_has_conflict", "Key-point conflict"),
    ],
)
agreement = pd.concat([review_agreement, keypoint_agreement], ignore_index=True)
if agreement.empty:
    print("Human labels have not yet been entered.")
else:
    display(agreement.style.format({"Exact agreement": "{:.1%}"}))
    agreement.to_csv(TABLES / f"human_judge_agreement_{SPLIT}.csv", index=False)



## Evidence-excerpt validation audit

The judge is asked to copy exact report passages for auditability. A non-verbatim excerpt does not change the assigned alignment label, but it is flagged and requires manual checking. These warnings are not counted as negative report outcomes.


In [ ]:
warning_rows = []
analysis_pmids = set(analysis_reviews["pmid"])
for record in records:
    pmid = str(record["pmid"])
    if pmid not in analysis_pmids:
        continue
    for warning in record.get("evidence_validation_warnings", []):
        warning_rows.append({
            "pmid": pmid,
            "subject_model": record["subject_model"],
            "scope": warning.get("scope"),
            "key_point_id": warning.get("key_point_id", ""),
            "issue": warning.get("issue"),
            "invalid_items": " || ".join(warning.get("invalid_items", [])),
            "report_path": record.get("report_path", ""),
        })

evidence_warnings = pd.DataFrame(warning_rows)
if evidence_warnings.empty:
    print("No non-verbatim evidence excerpts in the analytical cohort.")
else:
    display(evidence_warnings)
    evidence_warnings.to_csv(
        TABLES / f"evidence_validation_warnings_{SPLIT}.csv", index=False
    )
    print("These cases require manual audit before final reporting.")


## Recommended thesis presentation

Use one main table with question relevance, macro KPA, core KPA, KPC, and
overconfidence, explicitly stating the paired sample size. Report full conclusion
and uncertainty distributions instead of reducing them to an arbitrary score.

The strongest main figure is the key-point status distribution because it
separates omission from contradiction. The combined categorical figure is
optional. Paired bootstrap comparisons, categories, difference types,
cross-tabulations, and the manual-review queue are better suited to diagnostics or
an appendix.

Until the cohort is complete, label the values preliminary. Exact-quote validation
failures are missing evaluations and must not be counted as negative outcomes.
